# Suite XRX — The Cpp RegExp dialect

`Expressions.Cpp.RegExp` models the regular expressions `std::regex` reads (ISO/IEC 14882, [re.grammar] and
[re.synopt]): its own grammar, ECMAScript's (ECMA-262, third edition, as [re.grammar] modifies it), and a `regex` that
names among its `syntax_option_type` flags one of the five grammars [re.synopt] selects by naming POSIX, whose pattern
is then a term of POSIX's dialect (`POSIX BRE` for `basic` and `grep`, `POSIX ERE` for `extended`, `awk` and `egrep`),
read as that dialect reads it. No subterm in C++ is one, a pattern being a string there: its holder does. Its kinds are
validated, bind nothing, cover C++'s three editions, print as ECMAScript or as POSIX writes them, are refused by
evaluation by name, and are checked complete against its grammar kept as data.

In [ ]:
import { readFileSync as readGrammar } from "node:fs";
import { Framework } from "@mbse/expressions";
import * as IEEE754 from "@mbse/expressions/IEEE754/Expressions";
import * as ISOIEC11404 from "@mbse/expressions/ISOIEC11404/Expressions";
import * as CFamily from "@mbse/expressions/C/Expressions";
import * as CPreprocessor from "@mbse/expressions/C/Preprocessor";
import * as CTypes from "@mbse/expressions/C/Types";
import * as CppFamily from "@mbse/expressions/Cpp/Expressions";
import * as CppTypes from "@mbse/expressions/Cpp/Types";
import * as BRE from "@mbse/expressions/POSIX/BRE";
import * as ERE from "@mbse/expressions/POSIX/ERE";
import * as CppRegExp from "../src/Cpp/RegExp/index.js";
import { Evaluators as FE, Standards as FS, Symbolics as SY, Terms as F } from "@mbse/expressions/Framework";
import { Builders, Errors, JSON, Validators } from "@mbse/schemas/Framework";
import { assert, equal, raises, same } from "./support.js";

const { NotImplementedError, ValueError } = Errors;
const [PP, CPP, XT, XR, XRText] = [CPreprocessor.Expressions, CppFamily.Expressions, CppTypes.Expressions, CppRegExp.Expressions, CppRegExp.Text];
const [B, R] = [BRE.Expressions, ERE.Expressions];

const store: any = new Builders.Store(); // the C family's dialects and POSIX's, Cpp RegExp's install first: the installs run in any order
for (const install of [Framework.install, ISOIEC11404.install, IEEE754.install, CppRegExp.install, CFamily.install, CppFamily.install,
  CTypes.install, CppTypes.install, CPreprocessor.install, BRE.install, ERE.install]) {
  store.transaction(install);
}
store.transaction(CppRegExp.define); // its editions

/** The term value `spec` writes, the root of `dialect`'s holder, written in a transaction. */
function term(spec: unknown, dialect: F.Dialect = XR.DIALECT): any {
  return F.term(store.transaction(() => dialect.path.split(".").reduce((at: any, part) => at[part], store.Expressions)
    .Expression((e: any) => e.root(dialect.spec(spec)))));
}

const validate = (spec: unknown): string[] => XR.DIALECT.validate(term(spec));
const text = (spec: unknown): string => XRText.ToText(term(spec));
const union = (name: string): string[] => store.registered(name).branches.map((b: any) => b.name); // a union's branches
/** A character, of its code point, and its spelling. */
const char = (c: string, spelling?: string) => (u: any) => u.character(
  (r: any) => spelling === undefined ? r.value(BigInt(c.codePointAt(0) as number)) : r.raw(spelling).value(BigInt(c.codePointAt(0) as number)));
const alt = (...elements: unknown[]) => (u: any) => u.alternative((r: any) => r.elements(elements));
const pattern = (...alternatives: unknown[]) => (u: any) => u.pattern((r: any) => r.alternatives(alternatives));
const capture = (...alternatives: unknown[]) => (u: any) => u.capturinggroup((r: any) => r.alternatives(alternatives));
const group = (...alternatives: unknown[]) => (u: any) => u.group((r: any) => r.alternatives(alternatives));

/** A quantifier: `element{least,most}`, lazy where not `greedy`, as `spelling` writes it where given. */
function repeat(element: unknown, least: bigint, most?: bigint, greedy?: boolean, spelling?: string): unknown {
  return (u: any) => u.quantifier((r: any) => {
    r = r.min(least).element(element);
    r = most === undefined ? r : r.max(most);
    r = greedy === undefined ? r : r.greedy(greedy);
    return spelling === undefined ? r : r.raw(spelling);
  });
}

const cls = (elements: unknown[], negate = false) => (u: any) => u.characterclass((r: any) => r.negate(negate).elements(elements));
const span = (low: unknown, high: unknown) => (u: any) => u.characterclassrange((r: any) => r.min(low).max(high));
const chars = (kind: string, negate = false) => (u: any) => u.characterset((r: any) => r.setKind(kind).negate(negate));
const anchor = (kind: string, negate = false) => (u: any) => u.assertion((r: any) => r.assertionKind(kind).negate(negate));
/** `(?=...)`, `(?!...)`. */
const ahead = (alternatives: unknown[], negate = false) => (u: any) => u.assertion(
  (r: any) => r.assertionKind("lookahead").negate(negate).alternatives(alternatives));
const backref = (n: bigint) => (u: any) => u.backreference((r: any) => r.ref(n));
const named = (kind: string, name: string) => (u: any) => u[kind]((r: any) => r.name(name)); // `[:alpha:]`, `[.a.]`, `[=a=]`
/** A regex, its pattern a term of `dialect`. */
const regex = (flags: string[], spec: unknown, dialect: F.Dialect = XR.DIALECT) => (u: any) => u.regex(
  (r: any) => r.flags(flags).pattern(dialect.spec(spec)));
const error = (source: string) => (u: any) => u.error((r: any) => r.text(source));
const missing = (kind: string) => (u: any) => u.missing((r: any) => r.kind(kind));
const abc = pattern(alt(char("a"), char("b")), alt(char("c"))); // `ab|c`
// POSIX's: a set's kind, an assertion's and a list of patterns, as its dialects write them
const posixDot = (u: any) => u.characterset((r: any) => r.setKind("any"));
const posixAnchor = (kind: string) => (u: any) => u.assertion((r: any) => r.assertionKind(kind));
const patterns = (...specs: unknown[]) => (u: any) => u.patternlist((r: any) => r.patterns(specs));

<a id="XRX-01"></a>
## XRX-01 · Its union and holder: its kinds and invalid syntax, a branch of the framework's terms, reading its family's dialects and POSIX's; a holder refining the framework's, round-tripped through JSON

In [ ]:
{
  assert(same(XR.TAGS, ["regex", "pattern", "alternative", "group", "capturinggroup", "quantifier", "characterclass",
    "characterclassrange", "characterset", "character", "assertion", "backreference", "exclass", "collatingelement",
    "equivalence"]) && XR.HELPERS.length === 0 && XR.SINCE.size === 0);
  assert(same([...XR.DIALECT.kinds().keys()], union(XR.DIALECT.union)) && same(union(XR.DIALECT.union), [...XR.TAGS, "error", "missing"]));
  assert(XR.DIALECT.name() === "Cpp RegExp" && XR.DIALECT.tag === "cpp_regexp" && XR.DIALECT.union === "Expressions.Cpp.RegExp.Term");
  assert(union("Expressions.Framework.Term").includes("cpp_regexp") && union("Expressions.Framework.Dialect").includes("cpp_regexp"));
  assert(same(XR.DIALECT.reads(), [CPP.DIALECT, XT.DIALECT, PP.DIALECT, B.DIALECT, R.DIALECT])); // its family's, then POSIX's
  assert(!XR.DIALECT.reads().slice(0, 3).some((dialect) => XR.TAGS.some((tag) => union(dialect.union).includes(tag)))); // no subterm is one
  const Holder: any = store.registered("Expressions.Cpp.RegExp.Expression");
  assert(Holder.ref && Holder.refines === store.registered("Expressions.Framework.Expression") && F.isExpression(Holder));
  assert(Holder.properties.get("root").schema === store.registered(XR.DIALECT.union) && same([...Holder.adjacencies.keys()], ["abi"]));
  const holder = store.transaction(() => store.Expressions.Cpp.RegExp.Expression((e: any) => e.root(XR.DIALECT.spec(
    regex(["extended", "icase"], abc, R.DIALECT)))));
  assert(Validators.Validate(store)(Holder, holder).length === 0 && XR.DIALECT.validate(holder).length === 0);
  assert(F.dialectOf(F.term(holder)) === XR.DIALECT.union && F.kindOf(F.term(holder))[0] === "regex");
  assert(F.same((JSON.FromJSON(store)(Holder, JSON.ToJSON(store)(Holder, holder)) as any).root, holder.root));
  raises(Error, () => store.transaction(() => // its root holds its own terms
    store.Expressions.Cpp.RegExp.Expression((e: any) => e.root(XR.DIALECT.spec(CPP.binary("+", CPP.identifier("x"), 1n))))),
  "'binaryexpression' is not a member of a union value");
  assert(XR.DIALECT.validate(term(CPP.identifier("x"), CPP.DIALECT), ["x"]).length === 0); // a C++ term, as C++ reads it
}

<a id="XRX-02"></a>
## XRX-02 · Its kinds: their properties in order, each term property its Term, a regex's pattern any dialect's; forms and make round-trip

In [ ]:
{
  const props = (name: string): string[] => [...store.registered(`Expressions.Cpp.RegExp.${name}`).properties.keys()];
  assert(same(props("Regex"), ["flags", "pattern"]));
  assert(same(props("Pattern"), ["alternatives"]) && same(props("Group"), ["alternatives"]) && same(props("CapturingGroup"), ["alternatives"])
    && same(props("Alternative"), ["elements"]));
  assert(same(props("Quantifier"), ["raw", "min", "max", "greedy", "element"]) && same(props("CharacterClass"), ["negate", "elements"]));
  assert(same(props("CharacterClassRange"), ["min", "max"]) && same(props("CharacterSet"), ["setKind", "negate"]));
  assert(same(props("Character"), ["raw", "value"]) && same(props("Assertion"), ["assertionKind", "negate", "alternatives"]));
  assert(same(props("Backreference"), ["ref"]) && same(props("ExClass"), ["name"]) && same(props("CollatingElement"), ["name"])
    && same(props("Equivalence"), ["name"]));
  const prop = (name: string, member: string): any => store.registered(`Expressions.Cpp.RegExp.${name}`).properties.get(member).schema;
  const Term = store.registered(XR.DIALECT.union);
  assert(prop("Regex", "pattern") === store.registered("Expressions.Framework.Term")); // the dialect its grammar names
  assert(prop("Quantifier", "element") === Term && prop("CharacterClassRange", "max") === Term);
  assert(prop("Pattern", "alternatives").element === Term && prop("Alternative", "elements").element === Term
    && prop("Assertion", "alternatives").element === Term);
  const made = F.form(term(regex(["basic"], pattern(alt(repeat(char("a"), 2n, 3n), backref(1n))), B.DIALECT)));
  assert(made.kind === "regex" && same((made.attributes.get("flags") as any).values, ["basic"])
    && same(made.arguments.map((a) => F.kindOf(a)[0]), ["pattern"]));
  assert(F.dialectOf(made.arguments[0]) === B.DIALECT.union); // a POSIX BRE term
  const quantified = F.form(term(repeat(cls([span(char("a"), char("z"))]), 1n, undefined, false)));
  assert(equal(quantified.attributes, new Map<string, unknown>([["min", 1n], ["greedy", false]]))
    && same(quantified.arguments.map((a) => F.kindOf(a)[0]), ["characterclass"]));
  for (const spec of [regex(["grep"], patterns(abc, pattern(alt(posixDot))), B.DIALECT), ahead([alt(anchor("word", true))], true),
    cls([named("exclass", "alpha"), named("collatingelement", "ch"), named("equivalence", "e"), chars("digit")], true)]) {
    assert(F.same(term(XR.DIALECT.make(F.form(term(spec)))), term(spec)));
  }
}

<a id="XRX-03"></a>
## XRX-03 · Its checks: what each kind's terms may be, a quantifier repeating an atom alone, and its attributes' values

In [ ]:
{
  const cases: [unknown, string[]][] = [
    [regex([], alt(char("a"))), ["a regex's pattern cannot hold an alternative"]],
    [pattern(char("a")), ["a pattern's alternatives cannot hold a character"]],
    [alt(pattern(alt())), ["an alternative's elements cannot hold a pattern"]],
    [group(alt(named("exclass", "alpha"))), ["argument 0: an alternative's elements cannot hold an exclass"]],
    [capture(char("a")), ["a capturinggroup's alternatives cannot hold a character"]],
    [repeat(alt(), 0n), ["a quantifier's element cannot hold an alternative"]],
    [repeat(repeat(char("a"), 1n), 0n), ["a quantifier's element cannot hold a quantifier"]], // Term: Atom Quantifier
    [repeat(anchor("word"), 0n), ["a quantifier's element cannot hold an assertion"]],
    [repeat(ahead([alt(char("a"))]), 0n, 1n), ["a quantifier's element cannot hold an assertion"]],
    [cls([alt()]), ["a characterclass's elements cannot hold an alternative"]],
    [span(chars("digit"), named("collatingelement", "z")), ["a characterclassrange's min cannot hold a characterset"]],
    [ahead([char("a")]), ["an assertion's alternatives cannot hold a character"]],
    [repeat(char("a"), -1n), ["a quantifier's min cannot be negative, got -1"]],
    [repeat(char("a"), 3n, 2n), ["a quantifier's max cannot be less than its min, got 2 and 3"]],
    [repeat(chars("word"), 0n), []], // `\w*`: a word characterset, which a quantifier repeats
    [(u: any) => u.quantifier((r: any) => r.element(char("a"))), ["a quantifier needs a min"]],
    [(u: any) => u.quantifier((r: any) => r.min(1n)), ["a quantifier needs an element"]],
    [chars("alpha"), ["CharacterSet.setKind cannot be 'alpha'"]],
    [chars("any", true), ["an any characterset cannot be negated"]],
    [(u: any) => u.characterset((r: any) => r), ["a characterset needs a setKind"]],
    [(u: any) => u.character((r: any) => r.value(0x110000n)), ["a character's value must be a code point, got 1114112"]],
    [(u: any) => u.character((r: any) => r.raw("a")), ["a character needs a value"]],
    [anchor("both"), ["Assertion.assertionKind cannot be 'both'"]],
    [anchor("start", true), ["a start assertion cannot be negated"]],
    [(u: any) => u.assertion((r: any) => r.assertionKind("end").alternatives([alt()])), ["an end assertion holds no alternatives"]],
    [(u: any) => u.assertion((r: any) => r.negate(true)), ["an assertion needs an assertionKind"]],
    [backref(0n), ["a backreference's ref must be positive, got 0"]],
    [named("exclass", "a:b"), ["an exclass's name cannot hold ':'"]],
    [named("collatingelement", "a.="), ["a collatingelement's name cannot hold '.'"]],
    [named("equivalence", ""), ["an equivalence needs a name"]],
  ];
  for (const [spec, problems] of cases) assert(same(validate(spec), problems), `${validate(spec)} != ${problems}`);
  for (const spec of [abc, alt(), cls([span(char("a"), named("collatingelement", "z"))]), repeat(group(alt(char("a"))), 0n, 1n),
    cls([chars("word", true), char("-"), named("equivalence", "e")]), pattern(alt(char("a"), anchor("start"), backref(12n)))]) {
    assert(validate(spec).length === 0);
  }
}

<a id="XRX-04"></a>
## XRX-04 · Its grammars: a regex's flags, its one grammar element, ECMAScript by default; each grammar's pattern a term of its dialect, POSIX's read as POSIX reads them, what C++ selects of them checked

In [ ]:
{
  assert(same(XR.GRAMMARS, ["ECMAScript", "basic", "extended", "awk", "grep", "egrep"]) && XR.ECMASCRIPT === "ECMAScript");
  assert(same(XR.SYNTAX_OPTIONS, ["icase", "nosubs", "optimize", "collate", ...XR.GRAMMARS, "multiline"]));
  assert(XR.grammar(["icase", "awk"]) === "awk" && XR.grammar(["icase"]) === "ECMAScript" && XR.grammar([]) === "ECMAScript"); // the default
  assert(equal(XR.GRAMMAR_DIALECTS, new Map([["ECMAScript", "Cpp.RegExp"], ["basic", "POSIX.BRE"], ["extended", "POSIX.ERE"],
    ["awk", "POSIX.ERE"], ["grep", "POSIX.BRE"], ["egrep", "POSIX.ERE"]])));
  const unflagged = (u: any) => u.regex((r: any) => r.pattern(XR.DIALECT.spec(abc))); // a regex with no flags, ECMAScript's
  assert(validate(unflagged).length === 0 && text(unflagged) === "ab|c");
  assert(same(validate(regex(["icase", "verbose", "basic"], abc, B.DIALECT)), ["'verbose' is not a syntax_option_type element",
    "alternation is not in the basic grammar"]));
  assert(same(validate(regex(["basic", "extended"], abc, B.DIALECT)), ["a regex has at most one grammar element, got basic, extended"]));
  assert(same(validate(regex(["basic"], missing("pattern"), B.DIALECT)), ["pattern: missing pattern"]));
  assert(same(validate((u: any) => u.regex((r: any) => r.flags(["grep"]))), ["a regex needs a pattern"]));
  for (const [grammar, dialect] of [["basic", B.DIALECT], ["grep", B.DIALECT], ["extended", R.DIALECT], ["awk", R.DIALECT],
    ["egrep", R.DIALECT], ["ECMAScript", XR.DIALECT]] as [string, F.Dialect][]) {
    for (const other of [B.DIALECT, R.DIALECT, XR.DIALECT].filter((d) => d !== dialect)) {
      assert(same(validate(regex([grammar], pattern(alt(char("a"))), other)), [
        `the ${grammar} grammar's pattern is ${F.article(dialect.name())} term, got ${F.article(other.name())} term`]));
    }
    assert(validate(regex([grammar], pattern(alt(char("a"))), dialect)).length === 0);
  }
  assert(same(validate(regex(["basic"], alt(char("a")), B.DIALECT)), ["a regex's pattern cannot hold an alternative"]));
  assert(same(validate(regex([], patterns(abc), B.DIALECT)), ["the ECMAScript grammar's pattern is a Cpp RegExp term, got a POSIX BRE term"]));
  assert(equal(XR.FEATURES, new Map([["a patternlist", ["grep", "egrep"]], ["alternation", ["extended", "awk", "egrep"]],
    ["a lazy quantifier", []]])));
  assert(validate(regex(["grep"], patterns(pattern(alt(char("a"))), pattern(alt(posixAnchor("start")))), B.DIALECT)).length === 0);
  assert(validate(regex(["egrep"], patterns(abc), R.DIALECT)).length === 0 && same(validate(regex(["awk"], patterns(abc), R.DIALECT)), [
    "a patternlist is not in the awk grammar"]));
  assert(same(validate(regex(["grep"], patterns(abc), B.DIALECT)), ["alternation is not in the grep grammar"])); // POSIX.1-2024's `\|`
  assert(same(validate(regex(["extended"], pattern(alt(repeat(char("a"), 0n, undefined, false))), R.DIALECT)), [
    "a lazy quantifier is not in the extended grammar"])); // POSIX.1-2024's repetition modifier
  assert(same(validate(regex(["basic"], pattern(alt(char("a"), posixAnchor("start"))), B.DIALECT)), [
    "pattern: argument 0: a start assertion stands only where a BRE begins"])); // as POSIX BRE reads it
  assert(validate(regex(["extended"], pattern(alt(repeat(repeat(char("a"), 1n), 0n, 1n), char("a"), posixAnchor("start"))), R.DIALECT)).length === 0);
  assert(same(validate(regex(["extended"], pattern(alt((u: any) => u.quantifier((r: any) => r.min(1n)))), R.DIALECT)), [
    "pattern: argument 0: argument 0: a quantifier needs an element"]));
  assert(same(validate(regex(["basic"], pattern(alt(repeat(missing("atom"), 1n), error("\\{"))), B.DIALECT)), [ // invalid syntax
    "pattern: argument 0: argument 0: element: missing atom", "pattern: argument 0: argument 1: invalid syntax '\\\\{'"]));
}

<a id="XRX-05"></a>
## XRX-05 · What it binds: nothing; a capturing group numbered, a backreference resolving it by its number

In [ ]:
{
  assert(!store.registered(XR.DIALECT.union).branches.some((b: any) => F.isBinder(b.schema)));
  const grouped = term(regex([], pattern(alt(capture(alt(char("a"))), backref(1n)))));
  assert(SY.free(grouped, XR.DIALECT).size === 0 && XR.DIALECT.validate(grouped).length === 0);
  assert(same([...F.walk(grouped)].map((t) => F.kindOf(t)[0]), ["regex", "pattern", "alternative", "capturinggroup", "alternative",
    "character", "backreference"]));
}

<a id="XRX-06"></a>
## XRX-06 · Invalid syntax: error and missing in its Term, validated, printed as written, refused by name

In [ ]:
{
  assert(same(validate(error("(?<")), ["invalid syntax '(?<'"]) && text(error("(?<")) === "(?<");
  assert(same(validate(missing("atom")), ["missing atom"]) && text(alt(char("a"), missing("atom"))) === "a");
  assert(same(validate(pattern(alt(char("a"), error("[")))), ["argument 0: argument 1: invalid syntax '['"]));
  assert(text(pattern(alt(char("a"), error("[z")))) === "a[z" && text(regex(["basic"], error("\\("), B.DIALECT)) === "\\(");
  raises(ValueError, () => text((u: any) => u.error((r: any) => r)), "Cpp RegExp error has no text to print");
  raises(ValueError, () => XRText.ToText(term(CPP.identifier("x"), CPP.DIALECT)), "Cpp idexpression is not a regular expression to print");
}

<a id="XRX-07"></a>
## XRX-07 · Its editions: C++'s three, each by its designation and common name, each with every kind

In [ ]:
{
  const regexp = store.Instances.Expressions.Cpp.RegExp;
  assert(equal([...FS.standards(regexp).values()].map((s) => FS.tags(s)), [
    ["ISO/IEC 14882:2017", "C++17"], ["ISO/IEC 14882:2020", "C++20"], ["ISO/IEC 14882:2024", "C++23"]]));
  assert(equal(FS.properties(FS.find(regexp, "C++17")!).get("revision"), ["year", 2017n]));
  const names = ["Regex", "Pattern", "Alternative", "Group", "CapturingGroup", "Quantifier", "CharacterClass", "CharacterClassRange",
    "CharacterSet", "Character", "Assertion", "Backreference", "ExClass", "CollatingElement", "Equivalence"];
  assert([...FS.standards(regexp).values()].every((e) => same(FS.kinds(e).map((s: any) => s.name),
    names.map((name) => `Expressions.Cpp.RegExp.${name}`))));
  assert(store.transaction(CppRegExp.define)[0].get("Cpp20") === FS.find(regexp, "C++20")); // run again, it writes nothing
}

<a id="XRX-08"></a>
## XRX-08 · Printing: every kind as ECMAScript writes it, its escapes, a spelling as written; a regex of a POSIX grammar as POSIX writes it, awk's with its escapes

In [ ]:
{
  const every = pattern(alt(anchor("start"), capture(alt(char("a")), alt(char("."))), repeat(char("b"), 0n), repeat(char("c"), 1n),
    repeat(char("d"), 0n, 1n), repeat(char("e"), 2n, 2n), repeat(char("f"), 2n), repeat(char("g"), 2n, 5n),
    cls([span(char("a"), char("z")), named("exclass", "digit"), named("collatingelement", "ch"), named("equivalence", "e"), char("]")], true),
    chars("any"), backref(1n), anchor("end")));
  assert(text(every) === text(regex([], every)) && text(every) === "^(a|\\.)b*c+d?e{2}f{2,}g{2,5}[^a-z[:digit:][.ch.][=e=]\\]].\\1$");
  const ecma = pattern(alt(group(alt(char("a"))), ahead([alt(char("b"))]), ahead([alt(char("c"))], true), anchor("word"),
    anchor("word", true), chars("digit"), chars("space", true), chars("word"), repeat(char("x"), 0n, undefined, false),
    repeat(char("y"), 1n, 3n, true)),
  alt(char("\n"), char("\x01"), char("\x7f"), char("é"), cls([char("^"), char("-"), char("\\"), char("\t"), char("a")])));
  assert(text(ecma) === "(?:a)(?=b)(?!c)\\b\\B\\d\\S\\wx*?y{1,3}|\\n\\x01\\x7fé[\\^\\-\\\\\\ta]");
  const posix = pattern(alt(posixAnchor("start"), capture(alt(char("a"))), repeat(char("b"), 1n), char("\n"), posixAnchor("end")));
  assert(text(regex(["basic"], posix, B.DIALECT)) === "^\\(a\\)b\\{1,\\}\n$" && text(regex(["grep"], posix, B.DIALECT)) === "^\\(a\\)b\\{1,\\}\n$");
  assert(text(regex(["extended"], posix, R.DIALECT)) === "^(a)b+\n$" && text(regex(["egrep"], posix, R.DIALECT)) === "^(a)b+\n$");
  assert(text(regex(["awk"], posix, R.DIALECT)) === "^(a)b+\\n$"); // awk's escapes (XCU awk)
  assert(text(regex(["egrep"], patterns(abc, pattern(alt(char("(")))), R.DIALECT)) === "ab|c\n\\(");
  assert(text(repeat(char("a"), 0n, undefined, undefined, "{0,}")) === "a{0,}" && text(char("A", "\\x41")) === "\\x41"); // as written
  assert(text(regex(["icase"], abc)) === "ab|c"); // its flags are basic_regex's constructor's other argument
  assert(XRText.ToText(term(pattern(alt(posixDot)), B.DIALECT)) === "."); // a POSIX term, as POSIX writes it
}

<a id="XRX-09"></a>
## XRX-09 · Evaluation: each kind refused by its dialect's name, a POSIX pattern's by POSIX's

In [ ]:
{
  const interpreter = new FE.Interpreter(XR.DIALECT, new Map(), { evaluated: new Set() }); // no kind evaluated, each refused by its dialect's name
  for (const spec of [regex([], abc), abc, char("a"), cls([]), backref(1n)]) {
    raises(NotImplementedError, () => interpreter.run(term(spec)), `Cpp RegExp ${F.kindOf(term(spec))[0]} is not evaluated`);
  }
  raises(NotImplementedError, () => interpreter.run(term(pattern(alt(char("a"))), B.DIALECT)), "POSIX BRE pattern is not evaluated");
  raises(ValueError, () => interpreter.run(term(error("("))), "Cpp RegExp error is not evaluated");
}

<a id="XRX-10"></a>
## XRX-10 · Completeness: grammars/cpp-regex.json maps every production of [re.grammar]'s grammar, and every kind is mapped

In [ ]:
{
  const grammar = globalThis.JSON.parse(readGrammar("../../grammars/cpp-regex.json", "utf8"));
  const nodes: any[] = grammar.nodes;
  assert(nodes.length === 32 && new Set(nodes.map((n) => `${n.source}|${n.node}`)).size === 32 && grammar.checked.includes("32 productions and tokens"));
  assert(nodes.every((n) => {
    const keys = Object.keys(n).filter((k) => k !== "note").sort().join(",");
    return keys === "kinds,node,source" || keys === "node,not,source";
  }));
  const mapped = new Set(nodes.flatMap((n) => n.kinds ?? []));
  assert(equal([...mapped].sort(), XR.TAGS.map((tag) => `RegExp.${tag}`).sort())); // every kind, exactly: invalid syntax the framework's
  const sources: string[] = nodes.map((n) => n.source);
  assert(same([...new Set(sources)], ["ISO/IEC 14882", "ECMA-262 3rd edition, 15.10.1, as [re.grammar] modifies it"]));
  const added = ["ClassAtomExClass", "ClassAtomCollatingElement", "ClassAtomEquivalence", "ClassName", "ClassNameCharacter"];
  assert(added.every((name) => nodes.some((n) => n.node === name))); // [re.grammar]'s own productions
  assert(nodes.find((n) => n.node === "syntax_option_type").not.includes("POSIX BRE and POSIX ERE")); // POSIX's, theirs
}